In [ ]:
# # Cell 1: Mount Google Drive and Setup
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from tqdm import tqdm
# import warnings
# warnings.filterwarnings('ignore')

# # Set random seeds for reproducibility
# import random
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# import torch.nn.functional as F

# np.random.seed(42)
# torch.manual_seed(42)
# random.seed(42)


# # Path to your MIDI files
# MIDI_PATH = '/content/drive/MyDrive/clean_midi'
# print(f"Files in directory: {os.listdir(MIDI_PATH)[:10]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in directory: ['Jessica Simpson', 'Jett', 'Jeffrey Osborne', 'Jennifer Warnes', 'Jennifer Lopez', 'Jennifer Rush', 'Jerry Mungo', 'Jerry Herman', 'Jerry Lee Lewis', 'Jeremy Jackson']


In [ ]:
import os

path = '/content/drive/MyDrive/clean_midi'

midi_files = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".mid") or file.endswith(".midi"):
            midi_files.append(os.path.join(root, file))

print("Total MIDI files found:", len(midi_files))
print(midi_files[:3])

Total MIDI files found: 17176
["/content/drive/MyDrive/clean_midi/Jessica Simpson/I Think I'm in Love With You.mid", '/content/drive/MyDrive/clean_midi/Jessica Simpson/Where You Are.mid', '/content/drive/MyDrive/clean_midi/Jett/I Hate Myself for Loving You.mid']


In [ ]:
import pretty_midi

def midi_to_tokens(file_path):
    try:
        midi = pretty_midi.PrettyMIDI(file_path)

        tokens = []

        for instrument in midi.instruments:
            for note in instrument.notes:
                tokens.append(("NOTE_ON", note.pitch))
                tokens.append(("NOTE_OFF", note.pitch))

        return tokens

    except:
        return None

In [ ]:
import numpy as np

SEQ_LEN = 200
VOCAB_SIZE = 128 + 128 # 128 for NOTE_ON pitches, 128 for NOTE_OFF pitches

def encode(tokens):
    if tokens is None:
        return None

    encoded_seq = []
    for event_type, pitch in tokens:
        if event_type == 'NOTE_ON':
            encoded_seq.append(pitch) # IDs 0-127 for NOTE_ON
        elif event_type == 'NOTE_OFF':
            encoded_seq.append(128 + pitch) # IDs 128-255 for NOTE_OFF
        # Assuming pitch is always 0-127 based on MIDI standards

    # Truncate or pad to SEQ_LEN
    encoded_seq = encoded_seq[:SEQ_LEN]
    if len(encoded_seq) < SEQ_LEN:
        # Pad with 0. For simplicity, we use 0 as padding.
        # A dedicated padding token (e.g., VOCAB_SIZE if VOCAB_SIZE is then incremented) would be ideal.
        encoded_seq += [0] * (SEQ_LEN - len(encoded_seq))

    return np.array(encoded_seq, dtype=np.int32)

In [ ]:
import os
import glob
import numpy as np
import pretty_midi
import tensorflow as tf
import matplotlib.pyplot as plt

from tqdm import tqdm
import tensorflow as tf

from tensorflow import keras
from keras import layers
from keras.models import Model

!pip install scikit-learn

In [ ]:
import numpy as np

def encode(tokens):
    if tokens is None:
        return None

    encoded_seq = []
    for event_type, pitch in tokens:
        if event_type == 'NOTE_ON':
            encoded_seq.append(pitch) # IDs 0-127 for NOTE_ON
        elif event_type == 'NOTE_OFF':
            encoded_seq.append(128 + pitch) # IDs 128-255 for NOTE_OFF
        # Assuming pitch is always 0-127 based on MIDI standards

    # Truncate or pad to SEQ_LEN
    encoded_seq = encoded_seq[:SEQ_LEN]
    if len(encoded_seq) < SEQ_LEN:
        # Pad with 0. For simplicity, we use 0 as padding.
        # A dedicated padding token (e.g., VOCAB_SIZE if VOCAB_SIZE is then incremented) would be ideal.
        encoded_seq += [0] * (SEQ_LEN - len(encoded_seq))

    return np.array(encoded_seq, dtype=np.int32)

In [ ]:
data = []
limit = 300   # SMALL DATASET

for f in midi_files:
    tokens = midi_to_tokens(f)
    x = encode(tokens)

    if x is not None:
        data.append(x)

    if len(data) >= limit:
        break

data = np.array(data)
print("Dataset shape:", data.shape)

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Dataset shape: (300, 200)


In [ ]:
import tensorflow as tf

dataset = tf.data.Dataset.from_tensor_slices(data)
dataset = dataset.shuffle(1000).batch(16).prefetch(tf.data.AUTOTUNE)

In [ ]:
from tqdm import tqdm
import tensorflow as tf

from tensorflow import keras
from keras import layers
from keras.models import Model

LATENT_DIM = 64

inputs = layers.Input(shape=(SEQ_LEN,))

x = layers.Embedding(VOCAB_SIZE, 128)(inputs)
x = layers.LSTM(128)(x)

z_mean = layers.Dense(LATENT_DIM)(x)
z_log_var = layers.Dense(LATENT_DIM)(x)

In [ ]:
import tensorflow as tf

def sampling(args):
    z_mean, z_log_var = args
    eps = tf.random.normal(shape=(tf.shape(z_mean)[0], LATENT_DIM))
    return z_mean + tf.exp(0.5 * z_log_var) * eps

z = layers.Lambda(sampling)([z_mean, z_log_var])

In [ ]:
decoder_input = layers.RepeatVector(SEQ_LEN)(z)

x = layers.LSTM(128, return_sequences=True)(decoder_input)

outputs = layers.TimeDistributed(
    layers.Dense(VOCAB_SIZE, activation='softmax')
)(x)

In [ ]:
encoder = Model(inputs, [z_mean, z_log_var, z])

In [ ]:
latent_inputs = layers.Input(shape=(LATENT_DIM,))

x = layers.RepeatVector(SEQ_LEN)(latent_inputs)
x = layers.LSTM(128, return_sequences=True)(x)

outputs_dec = layers.TimeDistributed(
    layers.Dense(VOCAB_SIZE, activation='softmax')
)(x)

decoder = Model(latent_inputs, outputs_dec)

In [ ]:
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:

            z_mean, z_log_var, z = self.encoder(data)
            recon = self.decoder(z)

            # Ensure recon_loss is a scalar by explicitly reducing it
            recon_loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(data, recon)
            )

            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )

            loss = recon_loss + 0.1 * kl_loss

        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        # Cast the argmax output to tf.int32 to match 'data' type
        acc = tf.reduce_mean(
            tf.cast(
                tf.equal(data, tf.cast(tf.argmax(recon, axis=-1), tf.int32)),
                tf.float32
            )
        )

        return {
            "loss": loss,
            "recon_loss": recon_loss,
            "kl_loss": kl_loss,
            "accuracy": acc
        }

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')

In [ ]:
vae.fit(dataset, epochs=20)

Epoch 1/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 12s 47ms/step - accuracy: 0.0000e+00 - kl_loss: 0.3901 - loss: 5.4073 - recon_loss: 5.3683
Epoch 2/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.0321 - kl_loss: 0.3928 - loss: 5.1059 - recon_loss: 5.0666
Epoch 3/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.0288 - kl_loss: 0.2129 - loss: 4.6575 - recon_loss: 4.6362
Epoch 4/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.0471 - kl_loss: 0.2679 - loss: 4.6124 - recon_loss: 4.5856
Epoch 5/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.0508 - kl_loss: 0.3473 - loss: 4.3446 - recon_loss: 4.3098
Epoch 6/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.0933 - kl_loss: 0.3511 - loss: 4.1487 - recon_loss: 4.1135
Epoch 7/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.0458 - kl_loss: 0.2355 - loss: 4.2402 - recon_loss: 4.2167
Epoch 8/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.0587 - kl_loss: 0.3523 - loss: 4.0862 - recon_loss: 4.0509
Epo

In [ ]:
import numpy as np

def generate():
    z = np.random.normal(size=(1, LATENT_DIM))
    output = decoder.predict(z)

    return np.argmax(output[0], axis=-1)

sample = generate()
print(sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
[197 197 197 197 197 197 197 197 197 197  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82  82
  82  82]
